In [ ]:
!pip install spacy sklearn
!python -m spacy download en_core_web_sm

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [4]:
import spacy
from sklearn.metrics import classification_report
import re

In [5]:
def load_conll_data(file_path):
    sentences = []
    sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line == "":
                if sentence:
                    sentences.append(sentence)
                    sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word = parts[0]
                    tag = parts[-1]
                    sentence.append((word, tag))

    return sentences

train_data = load_conll_data("train.txt")
valid_data = load_conll_data("valid.txt")
test_data = load_conll_data("test.txt")

print("Training sentences:", len(train_data))
print("Validation sentences:", len(valid_data))
print("Test sentences:", len(test_data))

Training sentences: 14987
Validation sentences: 3466
Test sentences: 3684


In [6]:
def rule_based_ner(sentence):
    entities = []

    for word in sentence:
        if word.istitle():
            entities.append((word, "POSSIBLE_ENTITY"))

    return entities

# Example
sample_sentence = [word for word, tag in train_data[0]]
print("Rule-Based Example:")
print(rule_based_ner(sample_sentence))

Rule-Based Example:
[]


In [7]:
nlp = spacy.load("en_core_web_sm")

In [8]:
def model_based_ner(text):
    doc = nlp(text)
    entities = []

    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities

sample_text = "Barack Obama visited Germany and Microsoft headquarters."

print("Model-Based Example:")
print(model_based_ner(sample_text))

Model-Based Example:
[('Barack Obama', 'PERSON'), ('Germany', 'GPE'), ('Microsoft', 'ORG')]


In [9]:
def highlight_entities(text):
    doc = nlp(text)

    for ent in doc.ents:
        text = text.replace(ent.text, f"[{ent.text} ({ent.label_})]")

    return text

print("Highlighted Text:")
print(highlight_entities(sample_text))

Highlighted Text:
[Barack Obama (PERSON)] visited [Germany (GPE)] and [Microsoft (ORG)] headquarters.


In [10]:
def evaluate_model(test_data):
    y_true = []
    y_pred = []

    for sentence in test_data:
        words = [word for word, tag in sentence]
        true_tags = [tag for word, tag in sentence]

        text = " ".join(words)
        doc = nlp(text)

        predicted_tags = ["O"] * len(words)

        for ent in doc.ents:
            ent_words = ent.text.split()
            for i in range(len(words)):
                if words[i:i+len(ent_words)] == ent_words:
                    predicted_tags[i] = "B-" + ent.label_
                    for j in range(1, len(ent_words)):
                        predicted_tags[i+j] = "I-" + ent.label_

        y_true.extend(true_tags)
        y_pred.extend(predicted_tags)

    print(classification_report(y_true, y_pred))

print("Evaluation on Test Data:")
evaluate_model(test_data)

Evaluation on Test Data:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

               precision    recall  f1-score   support

   B-CARDINAL       0.00      0.00      0.00         0
       B-DATE       0.00      0.00      0.00         0
      B-EVENT       0.00      0.00      0.00         0
        B-FAC       0.00      0.00      0.00         0
        B-GPE       0.00      0.00      0.00         0
   B-LANGUAGE       0.00      0.00      0.00         0
        B-LAW       0.00      0.00      0.00         0
        B-LOC       0.57      0.02      0.04      1668
       B-MISC       0.00      0.00      0.00       702
      B-MONEY       0.00      0.00      0.00         0
       B-NORP       0.00      0.00      0.00         0
    B-ORDINAL       0.00      0.00      0.00         0
        B-ORG       0.53      0.37      0.43      1661
        B-PER       0.00      0.00      0.00      1617
    B-PERCENT       0.00      0.00      0.00         0
     B-PERSON       0.00      0.00      0.00         0
    B-PRODUCT       0.00      0.00      0.00         0
   B-QUAN

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
def save_predictions(test_data, filename="predictions.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for sentence in test_data:
            words = [word for word, tag in sentence]
            text = " ".join(words)
            doc = nlp(text)

            f.write("Sentence: " + text + "\n")
            for ent in doc.ents:
                f.write(f"{ent.text} - {ent.label_}\n")
            f.write("\n")

save_predictions(test_data)
print("Predictions saved!")

Predictions saved!
